# Q15：全部 bull flip 的 2015–2026 长样本稳健性

目标：在锁定信号、交易规则和 point-in-time 选池规则后，判断 2023–2026 的高 CAGR 能否跨越更长市场周期。

In [ ]:
from pathlib import Path
import os

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'xquant').exists():
            return candidate
    raise FileNotFoundError('xquant repository root not found')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'scan-v2-all-bull-flip-long-sample'
OUTPUT_DIR

## 1. Baseline 与实验边界

基准为 SPY 和沪深 300 ETF 买入持有。主要版本为 2015 冻结 PIT、月度滚动 PIT、原始 SuperTrend 翻多滚动 PIT。production breakout 固定为 ST 翻多且 `setup=breakout`，不要求 right_side_5 门槛。

In [ ]:
import pandas as pd

metrics = pd.read_csv(OUTPUT_DIR / 'portfolio_metrics.csv')
metrics[['strategy', 'cagr', 'annualizedVolatility', 'sharpe', 'calmar', 'maxDrawdown',
         'annualizedTurnover', 'averageGrossExposure', 'closedTrades']]

## 2. Q5：年度、月度与 benchmark 对照

In [ ]:
annual = pd.read_csv(OUTPUT_DIR / 'annual_metrics.csv')
monthly = pd.read_csv(OUTPUT_DIR / 'monthly_distribution.csv')
benchmark = pd.read_csv(OUTPUT_DIR / 'benchmark_comparison.csv')
display(annual.pivot(index='period', columns='strategy', values='return'))
display(monthly)
display(benchmark)

## 3. 回撤、恢复与滚动窗口

In [ ]:
from IPython.display import Image, display

display(Image(filename=OUTPUT_DIR / 'nav_and_drawdown.png'))
display(pd.read_csv(OUTPUT_DIR / 'drawdown_summary.csv'))
display(pd.read_csv(OUTPUT_DIR / 'rolling_window_summary.csv'))

## 4. 分市场、交易漏斗与资金占用

In [ ]:
display(pd.read_csv(OUTPUT_DIR / 'portfolio_metrics_by_market.csv'))
display(pd.read_csv(OUTPUT_DIR / 'execution_funnel.csv'))

## 5. 参数、成本和实现审计敏感性

In [ ]:
display(pd.read_csv(OUTPUT_DIR / 'supertrend_parameter_sensitivity.csv'))
display(pd.read_csv(OUTPUT_DIR / 'cost_sensitivity.csv'))
display(pd.read_csv(OUTPUT_DIR / 'audit_fix_long_sample_metrics.csv'))

## 6. 通俗结论

- 历史最好：月度滚动 PIT + production breakout，CAGR 21.09%，Sharpe 1.21。
- 风险调整：production breakout 的 Sharpe 较高；原始 ST 7/3 的 Calmar 略高。
- 结构上最值得继续：保持月度 PIT 与生产参数冻结，但按接近 40% 回撤和约三年恢复期做预算。
- 样本红利：2017、2019–2020、2025–2026 强趋势，以及当前仍可观察的美国/加密品种。
- 未来失效方式：横盘反复、跨市场同步下跌、高换手成本和历史候选目录幸存者偏差。
- 下一步：补齐真正的历史 security master、退市和 ETF 清盘记录后原参数重跑。